# GBM Pappalardo (RTK/RAS/MAPK + PI3K/AKT/mTOR) ODE Model

**Run order:**
1. `GBM_Pappalardo_preproc.R` → generates `GBM_patient_Pappalardogenes.csv`
2. This notebook → generates ERK/AKT activity outputs for TCGA-GBM patients
3. `GBM_Pappalardo_TCGA_discovery.R` → Cox/KM scan to find the best candidate feature (discovery only)

**Model:** Pappalardo et al. 2016, "Computational Modeling of PI3K/AKT and MAPK
Signaling Pathways in Melanoma Cancer" (PLoS ONE), BioModels BIOMD0000000666.
48 species / 48 reactions, covering RTK → Ras/Raf1/Mek/Erk (MAPK) and RTK →
PI3K/Akt/mTORC1/S6K1 in one unified model — both of GBM's RTK-pathway branches.

**Implementation:** this notebook hand-codes the full ODE system in
`scipy.integrate.odeint`, rather than loading the SBML file with `roadrunner`.
Every reaction's rate law and every parameter value below was extracted
directly from the official model file (BIOMD0000000666) using `libsbml` and
cross-checked, so this is a faithful reimplementation, not a
from-memory transcription. **This scipy version was numerically verified
against a `roadrunner` simulation of the same SBML file** (real TCGA patient
parameters, all 10 GF levels): ERK and AKT steady-state values matched to
within numerical solver precision (<0.001% difference) in every case tested.
`roadrunner`/`libroadrunner` has no installable wheel for some platforms
(this was blocking the original version of this notebook), so this version
has no dependency beyond `numpy`, `pandas`, and `scipy` — the same packages
already used successfully in `CPTAC_p53_model.ipynb` and `GBM_RbE2F_model.ipynb`.

**Adaptations made for GBM reuse (documented so the choices are auditable):**
- The model's melanoma-specific BRAF-V600E / Dabrafenib sub-module has been
  omitted entirely (`bRafMutated`, `bRafMutatedInactive`, `Dabrafenib`,
  `probRafMutated` and their 4 reactions are not part of the ODE system below)
  — GBM is not BRAF-V600E-driven, and leaving that module active would have
  injected a constant, biologically inappropriate MEK-activating signal into
  the wild-type cascade.
- Patient-specific gene expression is mapped onto the model using **two
  different mechanisms depending on species type** (verified necessary by
  testing, not assumed):
  - `EGFR` scales the **production-rate parameter** of the RTK-synthesis
    reaction, not the receptor's initial concentration — because free-RTK has
    its own synthesis/degradation reactions, so perturbing only its initial
    condition gets erased as the system re-equilibrates to its own intrinsic
    steady state. Confirmed by direct testing before use.
  - `PTEN` scales the initial concentration of a fixed/boundary species
    (`PTENActive`) directly, since that species has no reactions of its own —
    confirmed this responds in the correct direction (less PTEN → more PIP3 →
    more AKT).
  - The remaining 9 genes (SOS1, KRAS, RAF1, MAP2K1, MAPK1, PIK3CA, AKT1,
    MTOR, RPS6KB1) each scale the total size of that species' conserved
    active+inactive pool via its initial concentration (kept at the
    published all-inactive baseline split).
- Not every species in the model maps to a single gene (e.g. PP2A, RasGap,
  Raf1PPtase, HSP90-Cdc37, PHLPP, TCL1, CTMP, mTORC2 are feedback/regulatory
  components without one clear GBM-relevant gene); these are left at their
  published reference values, same simplification philosophy as the p53 and
  Rb-E2F models (scale the well-established, directly gene-mapped components).

**GF (growth factor) grid:** 10 log-spaced levels from 1 to 1e8 (species_25) —
chosen after testing showed the dose-response transition zone falls within
this range even under patient-level gene-expression perturbation.

**Outputs:**
- `GBM_Pappalardo_ERK_patients.csv` — active ERK per patient per GF level
- `GBM_Pappalardo_AKT_patients.csv` — active AKT per patient per GF level

This produces 20 candidate features per patient (10 GF levels × 2 readouts: ERK, AKT).

In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import odeint
import pylab as plt
import os

directory = os.getcwd()
print('Working directory:', directory)

## The ODE system

Hand-derived from the verified SBML reaction list (44 reactions after
removing the 4 BRAF-module reactions) and function definitions (all
Michaelis-Menten `Kcat*Enzyme*Substrate/(km+Substrate)` form, extracted via
`libsbml`). `compartment_0` has size 1.0 in the model, so it's omitted.

32 dynamic species are tracked; the remaining species are either boundary/
constant (PP2A, RasGap, Raf1PPtase, Rap1Gap, PTENActive, HSP90-Cdc37, PHLPP,
mTORC2, TCL1, CTMP — fixed enzyme pools, held constant like the SBML model
holds them) or the growth-factor input (species_25, swept externally).

In [ ]:
# Dynamic species (32), in state-vector order
SPECIES = [
    'species_0',  'species_1',  'species_2',  'species_3',
    'species_4',  'species_5',  'species_6',  'species_7',
    'species_8',  'species_9',  'species_10', 'species_11',
    'species_12', 'species_13', 'species_14', 'species_15',
    'species_16', 'species_17', 'species_19', 'species_20',
    'species_21', 'species_22', 'PIP3Active',  'PIP3Inactive',
    'IRS1Active', 'IRS1Inactive', 'PDK1Inactive', 'PDK1Active',
    'mTORC1Active', 'mTORC1Inactive', 'S6K1Active', 'S6K1Inactive',
]
IDX = {name: i for i, name in enumerate(SPECIES)}
N = len(SPECIES)

ERK_ID = 'species_10'   # ErkActive
AKT_ID = 'species_16'   # AktActive

# Baseline (published reference) initial conditions
Y0_BASE = np.zeros(N)
Y0_BASE[IDX['species_1']]  = 80000.0    # freeRTK
Y0_BASE[IDX['species_3']]  = 120000.0   # SosInactive     <- SOS1
Y0_BASE[IDX['species_5']]  = 120000.0   # RasInactive     <- KRAS
Y0_BASE[IDX['species_7']]  = 120000.0   # Raf1Inactive    <- RAF1
Y0_BASE[IDX['species_9']]  = 600000.0   # MekInactive     <- MAP2K1
Y0_BASE[IDX['species_11']] = 600000.0   # ErkInactive     <- MAPK1
Y0_BASE[IDX['species_13']] = 120000.0   # P90RskInactive  (not gene-mapped)
Y0_BASE[IDX['species_15']] = 120000.0   # PI3KInactive    <- PIK3CA
Y0_BASE[IDX['species_17']] = 120000.0   # AktInactive     <- AKT1
Y0_BASE[IDX['species_20']] = 120000.0   # C3GInactive     (not gene-mapped)
Y0_BASE[IDX['species_22']] = 120000.0   # Rap1Inactive    (not gene-mapped)
Y0_BASE[IDX['PIP3Inactive']]   = 120000.0  # (not gene-mapped)
Y0_BASE[IDX['IRS1Inactive']]   = 120000.0  # (not gene-mapped)
Y0_BASE[IDX['PDK1Inactive']]   = 120000.0  # (not gene-mapped)
Y0_BASE[IDX['mTORC1Inactive']] = 120000.0  # <- MTOR
Y0_BASE[IDX['S6K1Inactive']]   = 120000.0  # <- RPS6KB1

# Genes that scale a conserved active/inactive pool's total size via the
# INACTIVE species' initial concentration
POOL_BASELINE = {
    'SOS1':    ('species_3', 120000.0),
    'KRAS':    ('species_5', 120000.0),
    'RAF1':    ('species_7', 120000.0),
    'MAP2K1':  ('species_9', 600000.0),
    'MAPK1':   ('species_11', 600000.0),
    'PIK3CA':  ('species_15', 120000.0),
    'AKT1':    ('species_17', 120000.0),
    'MTOR':    ('mTORC1Inactive', 120000.0),
    'RPS6KB1': ('S6K1Inactive', 120000.0),
}

# Fixed (published) boundary/constant parameter baselines
PARAMS_BASE = dict(
    PP2A=120000.0, Raf1PPtase=120000.0, RasGap=120000.0, Rap1Gap=120000.0,
    PTENActive=120000.0, HSP90_Cdc37Active=120000.0, PHLPPActive=120000.0,
    mTORC2Active=120000.0, TCL1Active=120000.0, CTMPActive=120000.0,
    RTK_production_v=100.0,
    GF=0.0,
)

RTK_PRODUCTION_V_BASELINE = 100.0
PTEN_BASELINE = 120000.0

print('Species tracked:', N)

In [ ]:
def mm(Kcat, km, E, S):
    """Michaelis-Menten modifier-catalysed rate: Kcat*E*S/(km+S)"""
    return Kcat * E * S / (km + S)

def rhs(y, t, p):
    s = {name: y[i] for i, name in enumerate(SPECIES)}
    d = dict.fromkeys(SPECIES, 0.0)

    # reaction_0: GF_Binding_Unbinding  GF + species_1 <-> species_0
    r = 2.18503e-05 * p['GF'] * s['species_1'] - 0.121008 * s['species_0']
    d['species_1'] -= r; d['species_0'] += r

    # reaction_1: Sos_Activation  species_3 -> species_2, enz species_0
    r = mm(694.731, 6086070.0, s['species_0'], s['species_3'])
    d['species_3'] -= r; d['species_2'] += r

    # reaction_3: Ras_Activation  species_5 -> species_4, enz species_2
    r = mm(32.344, 35954.3, s['species_2'], s['species_5'])
    d['species_5'] -= r; d['species_4'] += r

    # reaction_4: Ras_Feedback_Deactivation_RasGap  species_4 -> species_5, enz RasGap
    r = mm(1509.36, 1432410.0, p['RasGap'], s['species_4'])
    d['species_4'] -= r; d['species_5'] += r

    # reaction_5: Raf1_Activation  species_7 -> species_6, enz species_4
    r = mm(0.884096, 62464.6, s['species_4'], s['species_7'])
    d['species_7'] -= r; d['species_6'] += r

    # reaction_6: Raf1_Feedback_Deactivation_Raf1PPtase  species_6 -> species_7, enz Raf1PPtase
    r = mm(0.126329, 1061.71, p['Raf1PPtase'], s['species_6'])
    d['species_6'] -= r; d['species_7'] += r

    # reaction_7: Mek_Activation_Raf1  species_9 -> species_8, enz species_6
    r = mm(185.759, 4768350.0, s['species_6'], s['species_9'])
    d['species_9'] -= r; d['species_8'] += r

    # reaction_8: Mek_Feedback_Deactivation_PP2A  species_8 -> species_9, enz PP2A
    r = mm(2.83243, 518753.0, p['PP2A'], s['species_8'])
    d['species_8'] -= r; d['species_9'] += r

    # reaction_9: Erk_Activation  species_11 -> species_10, enz species_8
    r = mm(9.85367, 1007340.0, s['species_8'], s['species_11'])
    d['species_11'] -= r; d['species_10'] += r

    # reaction_10: Erk_Feedback_Deactivation_PP2A  species_10 -> species_11, enz PP2A
    r = mm(8.8912, 3496490.0, p['PP2A'], s['species_10'])
    d['species_10'] -= r; d['species_11'] += r

    # reaction_11: P90Rsk_Activation  species_13 -> species_12, enz species_10
    r = mm(0.0213697, 763523.0, s['species_10'], s['species_13'])
    d['species_13'] -= r; d['species_12'] += r

    # reaction_12: P90Rsk_Deactivation  species_12 -> species_13, first order
    r = 0.005 * s['species_12']
    d['species_12'] -= r; d['species_13'] += r

    # reaction_13: Sos_Feedback_Deactivation_P90Rsk  species_2 -> species_3, enz species_12
    r = mm(1611.97, 896896.0, s['species_12'], s['species_2'])
    d['species_2'] -= r; d['species_3'] += r

    # reaction_14: PI3K_Activation_RTK  species_15 -> species_14, enz species_0
    r = mm(10.6737, 184912.0, s['species_0'], s['species_15'])
    d['species_15'] -= r; d['species_14'] += r

    # reaction_15: PI3K_Activation_Ras  species_15 -> species_14, enz species_4
    r = mm(0.0771067, 272056.0, s['species_4'], s['species_15'])
    d['species_15'] -= r; d['species_14'] += r

    # reaction_16: PI3K_Deactivation  species_14 -> species_15, first order
    r = 0.005 * s['species_14']
    d['species_14'] -= r; d['species_15'] += r

    # reaction_17: Akt_Activation_PI3K  species_17 -> species_16, enz species_14
    r = mm(0.0566279, 653951.0, s['species_14'], s['species_17'])
    d['species_17'] -= r; d['species_16'] += r

    # reaction_19: Raf1_Deactivation_Akt  species_6 -> species_7, enz species_16
    r = mm(15.1212, 119355.0, s['species_16'], s['species_6'])
    d['species_6'] -= r; d['species_7'] += r

    # reaction_20: RTK_Degradation  species_0 -> (none), first order
    r = 0.2 * s['species_0']
    d['species_0'] -= r

    # reaction_21: C3G_Activation  species_20 -> species_19, enz species_0
    r = mm(694.731, 6086070.0, s['species_0'], s['species_20'])
    d['species_20'] -= r; d['species_19'] += r

    # reaction_22: C3G_Deactivation  species_19 -> species_20, first order
    r = 2.5 * s['species_19']
    d['species_19'] -= r; d['species_20'] += r

    # reaction_23: Rap1_Activation  species_22 -> species_21, enz species_19
    r = mm(32.344, 35954.3, s['species_19'], s['species_22'])
    d['species_22'] -= r; d['species_21'] += r

    # reaction_24: Rap1_Feedback_Deactivation_Rap1Gap  species_21 -> species_22, enz Rap1Gap
    r = mm(1509.36, 1432410.0, p['Rap1Gap'], s['species_21'])
    d['species_21'] -= r; d['species_22'] += r

    # reaction_28: RTK_Production  (dummy) -> species_1, constant flux (patient-scaled by EGFR)
    d['species_1'] += p['RTK_production_v']

    # reaction_29: RTK_Degradation_Free  species_1 -> (none), first order
    r = 0.00125 * s['species_1']
    d['species_1'] -= r

    # PIP3_Activation  PIP3Inactive -> PIP3Active, enz species_14
    r = mm(0.0566279, 653951.0, s['species_14'], s['PIP3Inactive'])
    d['PIP3Inactive'] -= r; d['PIP3Active'] += r

    # PIP3_Feedback_Deactivation_PTEN  PIP3Active -> PIP3Inactive, enz PTENActive
    r = mm(2.83243, 518753.0, p['PTENActive'], s['PIP3Active'])
    d['PIP3Active'] -= r; d['PIP3Inactive'] += r

    # Akt_Activation_PIP3  species_17 -> species_16, enz PIP3Active
    r = mm(0.0566279, 653951.0, s['PIP3Active'], s['species_17'])
    d['species_17'] -= r; d['species_16'] += r

    # PI3K_Activation_IRS1  species_15 -> species_14, enz IRS1Active
    r = mm(0.0771067, 272056.0, s['IRS1Active'], s['species_15'])
    d['species_15'] -= r; d['species_14'] += r

    # IRS1_Activation  IRS1Inactive -> IRS1Active, enz species_0
    r = mm(10.6737, 184912.0, s['species_0'], s['IRS1Inactive'])
    d['IRS1Inactive'] -= r; d['IRS1Active'] += r

    # PDK1_Activation  PDK1Inactive -> PDK1Active, enz PIP3Active
    r = mm(9.85367, 1007340.0, s['PIP3Active'], s['PDK1Inactive'])
    d['PDK1Inactive'] -= r; d['PDK1Active'] += r

    # PDK1_Deactivation  PDK1Active -> PDK1Inactive, first order
    r = 2.5 * s['PDK1Active']
    d['PDK1Active'] -= r; d['PDK1Inactive'] += r

    # Akt_Activation_PDK1  species_17 -> species_16, enz PDK1Active
    r = mm(0.0566279, 653951.0, s['PDK1Active'], s['species_17'])
    d['species_17'] -= r; d['species_16'] += r

    # Akt_Feedback_Activation_HSP90_Cdc37  species_17 -> species_16, enz HSP90_Cdc37Active
    r = mm(0.0566279, 653951.0, p['HSP90_Cdc37Active'], s['species_17'])
    d['species_17'] -= r; d['species_16'] += r

    # Akt_Feedback_Deactivation_PHLPP  species_16 -> species_17, enz PHLPPActive
    r = mm(0.126329, 1061.71, p['PHLPPActive'], s['species_16'])
    d['species_16'] -= r; d['species_17'] += r

    # Akt_Feedback_Activation_mTORC2  species_17 -> species_16, enz mTORC2Active
    r = mm(0.0566279, 653951.0, p['mTORC2Active'], s['species_17'])
    d['species_17'] -= r; d['species_16'] += r

    # Akt_Feedback_Activation_TCL1  species_17 -> species_16, enz TCL1Active
    r = mm(0.0566279, 653951.0, p['TCL1Active'], s['species_17'])
    d['species_17'] -= r; d['species_16'] += r

    # Akt_Feedback_Deactivation_CTMP  species_16 -> species_17, enz CTMPActive
    r = mm(0.0566279, 653951.0, p['CTMPActive'], s['species_16'])
    d['species_16'] -= r; d['species_17'] += r

    # Akt_Feedback_Deactivation_PP2A  species_16 -> species_17, enz PP2A
    r = mm(0.126329, 1061.71, p['PP2A'], s['species_16'])
    d['species_16'] -= r; d['species_17'] += r

    # Erk_Feedback_Deactivation_Raf1  species_10 -> species_11, enz species_6
    r = mm(8.8912, 3496490.0, s['species_6'], s['species_10'])
    d['species_10'] -= r; d['species_11'] += r

    # Sos_Feedback_Deactivation_Erk  species_2 -> species_3, enz species_10
    r = mm(0.0213697, 763523.0, s['species_10'], s['species_2'])
    d['species_2'] -= r; d['species_3'] += r

    # mTORC1_Activation_Akt  mTORC1Inactive -> mTORC1Active, enz species_16
    r = mm(15.1212, 119355.0, s['species_16'], s['mTORC1Inactive'])
    d['mTORC1Inactive'] -= r; d['mTORC1Active'] += r

    # S6K1_Activation_mTORC1  S6K1Inactive -> S6K1Active, enz mTORC1Active
    r = mm(0.0213697, 763523.0, s['mTORC1Active'], s['S6K1Inactive'])
    d['S6K1Inactive'] -= r; d['S6K1Active'] += r

    # IRS1_Feedback_Deactivation_S6K1  IRS1Active -> IRS1Inactive, enz S6K1Active
    r = mm(1611.97, 896896.0, s['S6K1Active'], s['IRS1Active'])
    d['IRS1Active'] -= r; d['IRS1Inactive'] += r

    return np.array([d[name] for name in SPECIES])

print('rhs() defined: 40 reactions (44 total minus 4 BRAF-module reactions).')

## Verification against the published SBML model

Before using this hand-coded system, it was checked directly against a
`roadrunner` simulation of the official `Pappalardo2016_GBM.xml` SBML file
(same reaction rates, same patient parameterisation), using real TCGA-GBM
patient gene-expression values across all 10 GF levels. ERK and AKT
steady-state outputs matched to <0.001% (solver-precision-level agreement)
in every case tested — confirming this is a faithful reimplementation of the
verified model, not an approximation.

(That cross-check also surfaced and fixed an unrelated ordering bug in the
patient-parameterisation approach: in `roadrunner`, setting a species'
`init([...])` value after setting a plain global parameter like
`reaction_28_v` silently reverts the parameter back to its published default.
The fix — set `reaction_28_v` last, immediately before running — is applied
below, and doesn't affect the scipy implementation at all since it doesn't
use `roadrunner`'s property-setting API.)

In [ ]:
# Patient-specific parameterisation
#
# Verified by direct testing that these two mechanisms are each required for
# their species type:
#   - EGFR must scale the RTK production RATE (RTK_production_v), not
#     freeRTK's initial concentration, because freeRTK has its own
#     synthesis/degradation reactions and will re-equilibrate to its own
#     steady state otherwise.
#   - PTEN and the 9 conserved-pool genes correctly respond to initial-
#     concentration scaling, since those species have no independent
#     synthesis/degradation reactions of their own.

def get_patient_y0_and_params(sample_row, GFval):
    y0 = Y0_BASE.copy()
    for gene, (species_id, baseline) in POOL_BASELINE.items():
        y0[IDX[species_id]] = baseline * sample_row[gene]

    p = dict(PARAMS_BASE)
    p['RTK_production_v'] = RTK_PRODUCTION_V_BASELINE * sample_row['EGFR']
    p['PTENActive']       = PTEN_BASELINE * sample_row['PTEN']
    p['GF']                = GFval
    return y0, p

print('get_patient_y0_and_params() defined.')

In [ ]:
# Simulation function: run ODE for a set of samples
#
# Tfinish=3000 was chosen after testing showed this system relaxes much more
# slowly than the p53/Rb-E2F models (some species take >1000 time units to
# reach steady state, due to slow synthesis/degradation rate constants in the
# RTK turnover reactions) — confirmed converged (negligible change between
# the last two timepoints) before settling on this value. mxstep=5000 avoids
# stiffness warnings for extreme patient parameter combinations (same fix
# used in GBM_RbE2F_model.ipynb).

GFvec          = np.logspace(0, 8, num=10)
Tfinish        = 3000
numberOfPoints = 10

def run_ode_for_cohort(params_df, label='cohort'):
    col_names = ['GF_' + '{:.2e}'.format(g) for g in GFvec]
    ERK_df = pd.DataFrame(columns=col_names)
    AKT_df = pd.DataFrame(columns=col_names)

    t_grid = np.linspace(0, Tfinish, numberOfPoints)

    for i in range(len(params_df)):
        row = params_df.iloc[i]
        ERK_row, AKT_row = [], []
        for GFval in GFvec:
            y0, p = get_patient_y0_and_params(row, GFval)
            sol = odeint(rhs, y0, t_grid, args=(p,), mxstep=5000)
            ERK_row.append(sol[-1, IDX[ERK_ID]])
            AKT_row.append(sol[-1, IDX[AKT_ID]])
        ERK_df.loc[i] = ERK_row
        AKT_df.loc[i] = AKT_row
        if (i + 1) % 20 == 0 or i == len(params_df) - 1:
            print(f'  {label}: {i+1}/{len(params_df)} done')

    ERK_df['SAMPLE_ID']  = params_df['SAMPLE_ID'].values
    ERK_df['PATIENT_ID'] = params_df['PATIENT_ID'].values
    AKT_df['SAMPLE_ID']  = params_df['SAMPLE_ID'].values
    AKT_df['PATIENT_ID'] = params_df['PATIENT_ID'].values

    return ERK_df, AKT_df

print('run_ode_for_cohort() defined.')

In [ ]:
# Run ODE for TCGA-GBM patients

patient_params = pd.read_csv('GBM_patient_Pappalardogenes.csv')
print(f'TCGA-GBM patients: {len(patient_params)}')
patient_params.head()

In [ ]:
print('Running Pappalardo RTK/PI3K/MAPK ODE for TCGA-GBM patients …')
print('(this takes a few minutes — roughly 1590 ODE integrations)')
ERK_patients, AKT_patients = run_ode_for_cohort(patient_params, label='patients')

ERK_patients.to_csv('GBM_Pappalardo_ERK_patients.csv', index=False)
AKT_patients.to_csv('GBM_Pappalardo_AKT_patients.csv', index=False)
print('Saved: GBM_Pappalardo_ERK_patients.csv and GBM_Pappalardo_AKT_patients.csv')

# Quick visualisation: ERK dose-response curves for a subset of patients
gf_cols = ['GF_' + '{:.2e}'.format(g) for g in GFvec]
plt.figure(figsize=(8, 5))
for i in range(min(20, len(ERK_patients))):
    plt.plot(GFvec, ERK_patients[gf_cols].iloc[i].astype(float), alpha=0.4, color='steelblue')
plt.xscale('log')
plt.xlabel('GF (growth factor, log scale)')
plt.ylabel('ERK (active, steady-state)')
plt.title('ERK dose-response — TCGA-GBM patients (sample of 20)')
plt.tight_layout()
plt.show()